In [ ]:
import pandas as pd


In [ ]:
df=pd.read_csv("IMDB.csv",sep="\t")

In [ ]:
df

In [ ]:
df["sentiment"].value_counts()

In [ ]:
df.isna().sum()

Removing duplicates if any

In [ ]:
df.drop_duplicates(inplace=True)

PRE-PROCESSING

In [ ]:
# 1.CONVERTING INTO LOWERCASE
df["review"]=df["review"].str.lower()

In [ ]:
# 2.Removing the URLs
import re

def removeURLs(text):
    text=re.sub(r"http\S+","",text)
    return text

# 3.Removing the punctuations
def remove_punctuations(text):
    text=re.sub(r"[^a-zA-Z0-9\s]","",text)
    return text

# 4.Removing the html tags
def remove_html(text):
    text=re.sub(r"<.*?>","",text)
    return text

df["review"]=df["review"].apply(removeURLs)
df["review"]=df["review"].apply(remove_punctuations)
df["review"]=df["review"].apply(remove_html)

In [ ]:
df

Removing the stop words

In [10]:
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords


In [ ]:
def remove_stopwords(text):
    tokens=word_tokenize(text)
    stop_words=stopwords.words("english")
    tokenized_words=[word for word in tokens if word not in stop_words]
    return " ".join(tokenized_words)
df["review"]=df["review"].apply(remove_stopwords)

In [ ]:
df

Stemming


In [ ]:
from nltk.stem import PorterStemmer

In [ ]:
def stemming(text):
    ps=PorterStemmer()
    stemmed_words=[]
    tokens=word_tokenize(text)
    for token in tokens:
        stemmed_tokens=ps.stem(token)
        stemmed_words.append(stemmed_tokens)
    return " ".join(stemmed_words)
df["review"]=df["review"].apply(stemming)

In [ ]:
df.head()

7.Encoding

In [ ]:
from sklearn.preprocessing import LabelEncoder
le=LabelEncoder()
df["sentiment"]=le.fit_transform(df["sentiment"])

In [ ]:
y=df["sentiment"]

8. Vectorization

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
tf=TfidfVectorizer(max_features=5000)
X=tf.fit_transform(df["review"])

Datasets and DataLoader

In [ ]:
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

In [ ]:
import torch
from torch.utils.data import TensorDataset,DataLoader

In [ ]:
X_train=X_train.toarray()
X_test=X_test.toarray()

In [ ]:
train_set = TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(y_train.values).float()
)

test_set = TensorDataset(
    torch.from_numpy(X_test).float(),
    torch.from_numpy(y_test.values).float()
)

In [ ]:
train_loader = DataLoader(train_set, shuffle=True, batch_size=64)
test_loader = DataLoader(test_set, shuffle=True, batch_size=64)

In [ ]:
import torch.nn as nn
import torch.optim as optim

In [ ]:
class RNN(nn.Module):
    def __init__(self,input_size,hidden_size=128,num_layers=1):
        super().__init__()

        self.hidden_size=hidden_size
        self.num_layers=num_layers

        self.rnn=nn.RNN(input_size,hidden_size,num_layers,batch_first=True)

        self.fc=nn.Linear(hidden_size,1)
    def forward(self,x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size)

        out, _ = self.rnn(x, h0) 
        # 1st value = hidden state of all the timesteps => (batch, seq_len, hidden size)
        # 2nd value = final hidden state of last timestep

        out = self.fc(out[:, -1, :])
        return out

In [ ]:
input_size = X_train.shape[1]

model = RNN(input_size)

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters())


In [ ]:
epochs = 10

for epoch in range(epochs):
    model.train()

    for Xb, yb in train_loader:
        optimizer.zero_grad()

        Xb = Xb.unsqueeze(1) # add singleton direction
        
        outputs = model(Xb) # (batch_size, 1)

        outputs = torch.sigmoid(outputs.squeeze()) # (batch_size,) => probability

        loss = criterion(outputs, yb) # compute loss
        loss.backward() # backprop
        optimizer.step() # weights update

    print(f"epoch = {epoch+1}/{epochs} and loss = {loss.item()}")

In [ ]:
# evaluate

model.eval()

with torch.no_grad():
    correct_vals = 0
    tot_vals = 0
    
    for Xb, yb in test_loader:
        Xb = Xb.unsqueeze(1)

        outputs = model(Xb)
        predicted = (torch.sigmoid(outputs.squeeze()) > 0.5).float()

        tot_vals += yb.size(0)
        correct_vals += (predicted == yb).sum().item()

    print(f"accuracy = {correct_vals/tot_vals*100}")